<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [11]:

# ==========================================
# ⚙️ GLOBAL CONFIGURATION (TUNING)
# ==========================================
RSI_LEN = 12            # Lookback period for RSI
RSI_THRESH = 25         # Entry threshold (as defined in your backtest)
LOOKBACK_DAYS = 14      # How far back to search for trigger events
CSV_FILE = "OptionVolume.csv"  # Source for your ticker universe
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from {CSV_FILE}")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ {CSV_FILE} not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI {RSI_LEN} < {RSI_THRESH} triggers since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < RSI_LEN:
                continue

            # Clean columns for multi-index compatibility
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Calculate RSI
            df['RSI'] = calculate_rsi_precision(df['Close'], period=RSI_LEN)

            # IDENTIFY THE TRIGGER:
            # RSI crosses below the threshold today, having been above it yesterday
            df['Trigger'] = (df['RSI'] < RSI_THRESH) & (df['RSI'].shift(1) >= RSI_THRESH)

            # Filter for the lookback window
            recent_hits = df[df.index.date >= trigger_cutoff]
            recent_hits = recent_hits[recent_hits['Trigger'] == True]

            for date, row in recent_hits.iterrows():
                report_data.append({
                    "Date": date.strftime('%Y-%m-%d'),
                    "Symbol": symbol,
                    "Price": round(row['Close'], 2),
                    "RSI_Value": round(row['RSI'], 2)
                })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*60)
    print(f"🚀 RSI TRIGGER REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Config: RSI({RSI_LEN}) Threshold < {RSI_THRESH}")
    print("="*60)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)

        # Sort by Date descending (most recent first)
        results_df = results_df.sort_values(by="Date", ascending=False)

        # Display the detailed table
        print(results_df.to_string(index=False))

    print("="*60)

✅ Loaded 150 symbols from OptionVolume.csv
🔍 Scanning for RSI 12 < 25 triggers since 2026-03-05...

🚀 RSI TRIGGER REPORT: LAST 14 DAYS
Config: RSI(12) Threshold < 25
      Date Symbol  Price  RSI_Value
2026-03-18    UPS  96.84      24.18
2026-03-12    UPS  97.89      24.01
2026-03-12    NKE  54.13      22.89
2026-03-12     BX 102.12      24.29
2026-03-09    UPS  99.94      24.57
2026-03-05    FXI  35.58      22.23
